In [2]:
import pandas as pd
import numpy as np
import calendar
import os


In [3]:

# Lista degli impianti (nomi corrispondono al suffisso nei nomi file)
impianti = [
    "comuneimpianto",
    "comunelimitrofo1",
    "comunelimitrofo2",
    "comunelimitrofo3",
    "comunelimitrofo4"
]

# Directory dove si trovano i CSV
base_dir = "/home/damn/Documents/PROJECTS/GITHUB/HACKATON/Data4Econ_Hackaton/in-sample"

# Dizionario in cui salveremo i DataFrame degli errori medi per ciascun impianto
# Struttura: errori_per_impianto["comuneimpianto"] = DataFrame_errori (indice=mesi, colonne=variabili)
errori_per_impianto = {}

for impianto in impianti:
    # Costruisci i path relativi ai file CSV di predizione e reale
    pred_path = os.path.join(base_dir, f"2024_meteo_fcs_{impianto}.csv")
    real_path = os.path.join(base_dir, f"2024_meteo_act_{impianto}.csv")
    
    # Lettura dei CSV in DataFrame
    meteo_pred  = pd.read_csv(pred_path, sep=";")
    meteo_reale = pd.read_csv(real_path, sep=";")
    
    # 1) Assicuriamoci che "Dataora" sia datetime
    meteo_pred["Dataora"]  = pd.to_datetime(meteo_pred["Dataora"])
    meteo_reale["Dataora"] = pd.to_datetime(meteo_reale["Dataora"])
    
    # 2) Individua le colonne da confrontare (tutte tranne "Dataora")
    colonne_da_confrontare = [col for col in meteo_pred.columns if col != "Dataora"]
    
    # 3) Dizionario temporaneo per gli errori medi di questo impianto
    errori_medi_per_mese = {}
    
    # 4) Ciclo sui 12 mesi
    for mese in range(1, 13):
        nome_mese = calendar.month_name[mese].lower()  # "gennaio", "febbraio", ...
        
        # Filtra predetti e reali per anno 2024 e mese corrente
        df_pred = meteo_pred[
            (meteo_pred["Dataora"].dt.year  == 2024) &
            (meteo_pred["Dataora"].dt.month == mese)
        ].copy()
        
        df_reale = meteo_reale[
            (meteo_reale["Dataora"].dt.year  == 2024) &
            (meteo_reale["Dataora"].dt.month == mese)
        ].copy()
        
        # Inizializza il sotto‐dizionario per il mese
        errori_medi_per_mese[nome_mese] = {}
        
        for col in colonne_da_confrontare:
            # Controlla presenza colonna e non-vuoto
            if col not in df_pred.columns or col not in df_reale.columns:
                errore = np.nan
            else:
                # Assumo che le righe corrispondano in ordine cronologico;
                # se non fosse il caso, bisognerebbe fare un merge per Dataora.
                valori_pred  = df_pred[col].astype(float).values
                valori_reale = df_reale[col].astype(float).values
                
                if len(valori_pred) == 0 or len(valori_reale) == 0 or (len(valori_pred) != len(valori_reale)):
                    errore = np.nan
                else:
                    diff = np.abs(valori_pred - valori_reale)
                    N = len(diff)
                    
                    if col == "WDD":
                        # Errore medio per WDD: sum(abs(pred - reale) / 3.6) * (1/N)
                        diff = diff / 3.6
                        errore = diff.sum() * (1 / N)
                    else:
                        # Errore medio per le altre colonne: sum(abs(pred - reale)) / N
                        errore = diff.sum() / N
            
            # Memorizza l’errore per questa colonna/questo mese
            errori_medi_per_mese[nome_mese][col] = errore
            
            # (Opzionale) crea variabile globale per rintracciare facilmente
            var_name = f"errore_{col.lower()}_{impianto}_{nome_mese}"
            globals()[var_name] = errore
    
    # 5) Costruisci il DataFrame degli errori medi per questo impianto
    df_errori = pd.DataFrame.from_dict(
        errori_medi_per_mese,
        orient="index",
        columns=colonne_da_confrontare
    )
    df_errori.index.name = "mese"
    
    # 6) Salva il DataFrame nel dizionario finale
    errori_per_impianto[impianto] = df_errori



In [ ]:

# A questo punto:
# - Per ogni impianto (chiave di errori_per_impianto) hai un DataFrame (indice mesi, colonne RNF, RHM, WDS, WDD, ecc.).
# - Le variabili globali create (errore_<col>_<impianto>_<mese>) contengono il singolo valore di errore.
#
# # Esempio di utilizzo:
#
# Stampa dell’errore medio WDS per il mese di marzo dell’impianto "comuneimpianto":
# print(errore_wds_comuneimpianto_marzo)
#
# Visualizza l’intero DataFrame degli errori per l’impianto "comunelimitrofo2":
# print(errori_per_impianto["comunelimitrofo2"])


In [4]:
errori_per_impianto

{'comuneimpianto':                 WDS       WDD       RNF       RHM
 mese                                             
 january    0.661962  1.858274  0.021102  0.956720
 february   0.495977  1.292585  0.014943  0.764224
 march      0.663257  1.800359  0.020996  0.962450
 april      0.802500  2.167130  0.015694  1.364722
 may        0.766129  3.334677  0.039919  1.757796
 june       0.757361  3.445602  0.019167  1.575556
 july       0.574731  3.855697  0.010618  1.432258
 august     0.621640  5.010715  0.013575  1.563441
 september  0.891250  2.495332  0.036250  1.326806
 october    0.606855  2.543309  0.027016  1.039516
 november   0.435694  1.905633  0.012639  0.978611
 december   0.524328  1.903039  0.016801  0.799866,
 'comunelimitrofo1':                 WDS       WDD       RNF       RHM
 mese                                             
 january    0.611828  1.894041  0.020968  0.942339
 february   0.505747  1.351692  0.015230  0.861638
 march      0.669448  1.604045  0.022476  0